# Phase II_03: Metadata Extraction

**Objective**: Extract and map CaBuAr tile metadata (dates, coordinates)

**Input**: CaBuAr HDF5 from Google Drive cache

**Output**: JSON database with tile metadata

## Setup: Mount Google Drive

In [ ]:
%pip install -q h5py
import sys
import json
import numpy as np
from collections import defaultdict
from pathlib import Path
from datetime import datetime

sys.path.insert(0, '/content/RETINNA/src')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✓ Google Drive mounted")

# Setup drive cache
from drive_utils import setup_drive_for_colab

drive_manager = setup_drive_for_colab(verbose=True)

if drive_manager is None:
    raise RuntimeError("Failed to connect to Google Drive - aborting")

print(f"\n✓ Drive root: {drive_manager.root}")

# Find CaBuAr HDF5 in Drive cache
cabuaur_cache = drive_manager.root / 'cabuaur_cache'
hdf5_path = cabuaur_cache / '512x512.hdf5'

if hdf5_path.exists():
    print(f"✓ Found CaBuAr in Drive cache: {hdf5_path}")
else:
    print(f"⚠ CaBuAr not found at {hdf5_path}")
    # Try TorchGeo cache
    hdf5_path = Path('/tmp/cabuaur/512x512.hdf5')
    if hdf5_path.exists():
        print(f"  Using local cache instead: {hdf5_path}")
    else:
        raise FileNotFoundError("CaBuAr HDF5 not found in Drive or local cache")

print(f"\n✓ Using: {hdf5_path}")
print(f"✓ Dependencies loaded")

## Extract metadata from HDF5

In [ ]:
import h5py

def extract_cabuaur_metadata(hdf5_path):
    """
    Extract available metadata from CaBuAr HDF5 file.
    
    Returns:
        dict: Metadata for all samples
        dict: Fire map (UUID -> sample keys)
    """
    metadata = {}
    fire_map = defaultdict(list)
    
    with h5py.File(hdf5_path, 'r') as f:
        all_keys = list(f.keys())
        print(f"Total samples in HDF5: {len(all_keys)}")
        
        for idx, key in enumerate(all_keys):
            if (idx + 1) % 100 == 0:
                print(f"  Processed {idx + 1}/{len(all_keys)}...")
            
            sample = f[key]
            
            # Extract available metadata
            uuid = key.split('_')[0]
            tile_idx = int(key.split('_')[-1])
            fold = int(sample.attrs.get('fold', -1))
            comments = list(sample.attrs.get('comments', []))
            
            metadata[key] = {
                'uuid': uuid,
                'tile_idx': tile_idx,
                'fold': fold,
                'comments': comments,
                'fold_name': ['val', 'train', 'train', 'train', 'train'][fold] if fold >= 0 else 'unknown'
            }
            
            fire_map[uuid].append(key)
    
    return metadata, dict(fire_map)

# Extract metadata
print(f"Extracting metadata from {hdf5_path}...")
metadata, fire_map = extract_cabuaur_metadata(str(hdf5_path))

print(f"\n✓ Extracted {len(metadata)} samples")
print(f"✓ Found {len(fire_map)} unique fires")
print(f"\nFire statistics:")
tile_counts = [len(tiles) for tiles in fire_map.values()]
print(f"  Avg tiles per fire: {np.mean(tile_counts):.1f}")
print(f"  Min tiles: {np.min(tile_counts)}, Max tiles: {np.max(tile_counts)}")

## Build JSON database

In [ ]:
def build_metadata_database(metadata, fire_map):
    """
    Build structured JSON database of all samples.
    """
    database = {
        'metadata': {
            'created': datetime.now().isoformat(),
            'total_fires': len(fire_map),
            'total_samples': len(metadata),
            'note': 'UUIDs are fire identifiers; coordinate/date info to be added from external sources'
        },
        'fires': []
    }
    
    for uuid, sample_keys in fire_map.items():
        fire_entry = {
            'uuid': uuid,
            'fire_name': None,
            'fire_id': None,
            'pre_date': None,
            'post_date': None,
            'location': {
                'bbox_north': None,
                'bbox_south': None,
                'bbox_east': None,
                'bbox_west': None,
                'center_lat': None,
                'center_lon': None,
                'utm_zone': None
            },
            'num_tiles': len(sample_keys),
            'split': {'train': 0, 'val': 0},
            'samples': []
        }
        
        # Count splits
        for key in sample_keys:
            fold_name = metadata[key]['fold_name']
            if fold_name in fire_entry['split']:
                fire_entry['split'][fold_name] += 1
        
        # Add sample details
        for key in sorted(sample_keys):
            fire_entry['samples'].append({
                'sample_id': key,
                'tile_idx': metadata[key]['tile_idx'],
                'fold': metadata[key]['fold_name'],
                'comments': metadata[key]['comments']
            })
        
        database['fires'].append(fire_entry)
    
    return database

print("Building metadata database...")
database = build_metadata_database(metadata, fire_map)

print(f"\n✓ Database created:")
print(f"  Total fires: {database['metadata']['total_fires']}")
print(f"  Total samples: {database['metadata']['total_samples']}")
print(f"  First fire:")
fire_0 = database['fires'][0]
print(f"    UUID: {fire_0['uuid']}")
print(f"    Tiles: {fire_0['num_tiles']}")
print(f"    Split: {fire_0['split']}")

## Save to Google Drive

In [ ]:
# Save to Drive
output_file = 'cabuaur_metadata.json'

metadata_path = drive_manager.save_with_timestamp(
    database,
    rel_path='phase2/II_03_metadata',
    filename_base='cabuaur_metadata',
    file_format='.json',
    verbose=True
)

if metadata_path:
    print(f"\n✓ Metadata saved to Drive: {metadata_path.name}")
else:
    print(f"\n✗ Failed to save to Drive")

## Summary

In [ ]:
print("CaBuAr Metadata Database Summary")
print("=" * 70)

print(f"\nData extracted from HDF5:")
print(f"  ✓ Fire UUID (unique fire identifier)")
print(f"  ✓ Tile index within fire")
print(f"  ✓ Train/Val/Test split")
print(f"  ✓ Comments array (unknown encoding)")

print(f"\nData NOT in HDF5 (needs external source):")
print(f"  ✗ Acquisition dates (pre/post-fire)")
print(f"  ✗ Geographic coordinates")
print(f"  ✗ Fire names or incident IDs")

print(f"\nNext steps:")
print(f"  1. Query Sentinel-2 API to get acquisition dates")
print(f"  2. Map fire UUIDs to Cal Fire incident database")
print(f"  3. Extract bounding boxes from fire perimeters")
print(f"  4. Update JSON database with location/date info")

print(f"\nDatabase saved to Drive: phase2/II_03_metadata/")